<a href="https://colab.research.google.com/github/DanilMarinkovic/NeuralNetworksCW3/blob/nb1-prototype/notebooks/notebook_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
%pip install torch_geometric --upgrade --quiet

In [19]:
import torch
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GAT
from copy import deepcopy
import torch.nn.functional as F
import numpy as np

In [20]:
# Load Cora dataset
dataset = Planetoid(root="./data", name="Cora")
data = dataset[0]

# Use GPU if possible
device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device)

data = data.to(device)

# Split our data into 3 sets
train_mask = data.train_mask
val_mask = data.val_mask
test_mask = data.test_mask

torch 2.14.1+cpu | device: cpu


In [21]:
# GAT
def make_gat():
  return GAT(
      in_channels=dataset.num_features,    # 1433
      hidden_channels=8,
      num_layers=2,
      out_channels=dataset.num_classes,    # 7
      heads=8,
      dropout=0.6,
  ).to(device)

# GATv2
def make_gatv2():
  return GAT(
      in_channels=dataset.num_features,
      hidden_channels=8,
      num_layers=2,
      out_channels=dataset.num_classes,
      heads=8,
      dropout=0.6,
      v2=True,
  ).to(device)


In [22]:
@torch.no_grad()
def test(model, data, mask=None):
    mask = data.test_mask if mask is None else mask
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=-1)

    return (pred[mask] == data.y[mask]).float().mean().item()

def train_model(model, data, seed=0, lr=0.005, weight_decay=5e-4, epochs=200):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        val_acc = test(model, data, data.val_mask)
        if val_acc > best_val:
            best_val = val_acc
            best_state = deepcopy(model.state_dict())

        if epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | loss {loss.item():.4f} | "
                  f"train acc {val_acc:.3f}")

    model.load_state_dict(best_state)   # restore best-validation weights

    return model

In [23]:
# Train a model with a set seed
def run(model_fn, seed):
  torch.manual_seed(seed)
  # Fresh model
  model = model_fn().to(device)
  return train_model(model, data, seed=seed)

In [24]:
# Train our models
models = {"GAT": [], "GATv2": []}
for seed in range(3):
    print(f"Seed: {seed}")

    print("Model: GAT")
    models["GAT"].append(run(make_gat, seed))

    print("Model: GATv2")
    models["GATv2"].append(run(make_gatv2, seed))

Seed: 0
Model: GAT
Epoch 000 | loss 1.9451 | train acc 0.248
Epoch 020 | loss 1.7508 | train acc 0.510
Epoch 040 | loss 1.2831 | train acc 0.662
Epoch 060 | loss 0.9196 | train acc 0.770
Epoch 080 | loss 0.7745 | train acc 0.770
Epoch 100 | loss 0.5727 | train acc 0.766
Epoch 120 | loss 0.5469 | train acc 0.782
Epoch 140 | loss 0.5237 | train acc 0.764
Epoch 160 | loss 0.5144 | train acc 0.774
Epoch 180 | loss 0.4251 | train acc 0.774
Model: GATv2
Epoch 000 | loss 1.9466 | train acc 0.316
Epoch 020 | loss 1.7571 | train acc 0.676
Epoch 040 | loss 1.2469 | train acc 0.724
Epoch 060 | loss 0.9036 | train acc 0.740
Epoch 080 | loss 0.6582 | train acc 0.742
Epoch 100 | loss 0.5849 | train acc 0.752
Epoch 120 | loss 0.5680 | train acc 0.736
Epoch 140 | loss 0.4808 | train acc 0.740
Epoch 160 | loss 0.4840 | train acc 0.740
Epoch 180 | loss 0.5084 | train acc 0.738
Seed: 1
Model: GAT
Epoch 000 | loss 1.9444 | train acc 0.110
Epoch 020 | loss 1.7312 | train acc 0.544
Epoch 040 | loss 1.2707 |

In [25]:
# Test our models
for name, model_list in models.items():
    accs = [test(m, data, test_mask) for m in model_list]
    print(f"{name}: {np.mean(accs):.3f} ± {np.std(accs):.3f}")

GAT: 0.796 ± 0.006
GATv2: 0.781 ± 0.007
